In [2]:
import dagshub
dagshub.init(repo_owner='rahulpatel16092005', repo_name='youtube-comment-analysis', mlflow=True)

Accessing as rahulpatel16092005

Initialized MLflow to track repo "rahulpatel16092005/youtube-comment-analysis"

Repository rahulpatel16092005/youtube-comment-analysis initialized!

In [5]:
import mlflow
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from gensim.models import Word2Vec
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

In [6]:
df = pd.read_csv('preprocessed_data.csv').dropna(subset=['comment'])
df.shape

(36662, 5)

In [7]:
# Set or create an experiment
mlflow.set_experiment("Exp 2 - BoW vs TfIdf vs Word2vec")

2026/04/13 16:36:30 INFO mlflow.tracking.fluent: Experiment with name 'Exp 2 - BoW vs TfIdf vs Word2vec' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/d8665914ef7249c19b7069611368fc53', creation_time=1776078390535, experiment_id='1', last_update_time=1776078390535, lifecycle_stage='active', name='Exp 2 - BoW vs TfIdf vs Word2vec', tags={}, trace_location=None, workspace='default'>

In [8]:
# Function to vectorize data
def vectorize_data(X_train, X_test, vectorizer_type, ngram_range, max_features, vector_size=None):
    if vectorizer_type == 'bow':
        vectorizer = CountVectorizer(ngram_range=ngram_range, max_features=max_features)
    elif vectorizer_type == 'tfidf':
        vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
    elif vectorizer_type == 'word2vec':
        # Word2Vec specific process
        model = Word2Vec(sentences=X_train['comment'].apply(lambda x: x.split()), vector_size=vector_size, window=5, min_count=1)
        X_train_word2vec = np.array([np.mean([model.wv[word] for word in words if word in model.wv] or [np.zeros(vector_size)], axis=0) for words in X_train['comment'].apply(lambda x: x.split())])
        X_test_word2vec = np.array([np.mean([model.wv[word] for word in words if word in model.wv] or [np.zeros(vector_size)], axis=0) for words in X_test['comment'].apply(lambda x: x.split())])
        return X_train_word2vec, X_test_word2vec

    X_train_vec = vectorizer.fit_transform(X_train['comment']).toarray()
    X_test_vec = vectorizer.transform(X_test['comment']).toarray()

    return X_train_vec, X_test_vec

In [9]:
# Objective function for Optuna
def objective(trial):
    # Split data
    X = df[['comment', 'word_count', 'char_count', 'avg_word_length']]
    y = df['category']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Suggest hyperparameters
    vectorizer_type = trial.suggest_categorical("vectorizer_type", ["bow", "tfidf", "word2vec"])
    vector_size = None
    max_features = None
    ngram_range = None

    if vectorizer_type != 'word2vec':
        ngram_range_str = trial.suggest_categorical("ngram_range", ["(1, 1)", "(1, 2)", "(1, 3)"])
        ngram_range = eval(ngram_range_str)
        max_features = trial.suggest_int("max_features", 1000, 10000)
    else:
        vector_size = trial.suggest_int("vector_size", 100, 300)

    # Vectorize data
    X_train_vec, X_test_vec = vectorize_data(X_train, X_test, vectorizer_type, ngram_range, max_features, vector_size)

    # Combine additional features
    X_train_combined = np.hstack([X_train_vec, X_train[['word_count', 'char_count', 'avg_word_length']].values])
    X_test_combined = np.hstack([X_test_vec, X_test[['word_count', 'char_count', 'avg_word_length']].values])

    # Train a RandomForest model
    model = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42)
    model.fit(X_train_combined, y_train)

    # Make predictions
    y_pred = model.predict(X_test_combined)
    accuracy = accuracy_score(y_test, y_pred)

    # Log each run with MLflow
    with mlflow.start_run() as run:
        # Set run name
        if vectorizer_type == "word2vec":
            mlflow.set_tag("mlflow.runName", f"word2vec_{vector_size}")
        else:
            mlflow.set_tag("mlflow.runName", f"{ngram_range_str}_{vectorizer_type}_{max_features}")

        mlflow.log_param("vectorizer_type", vectorizer_type)
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("max_features", max_features)
        mlflow.log_param("vector_size", vector_size)

        # Log model metrics
        mlflow.log_metric("accuracy", accuracy)

        # Logging the classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):  # For precision, recall, f1-score, etc.
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Confusion matrix plot
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title("Confusion Matrix")

        # Save and log the confusion matrix plot
        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")

    return accuracy

In [ ]:
# Running Optuna
study = optuna.create_study(direction="maximize", study_name='Bow vs TFIDF vs Word2vec')
study.optimize(objective, n_trials=200)

[I 2026-04-13 16:36:58,742] A new study created in memory with name: Bow vs TFIDF vs Word2vec


🏃 View run word2vec_168 at: https://dagshub.com/rahulpatel16092005/youtube-comment-analysis.mlflow/#/experiments/1/runs/9c7993a75a084ca8ad9b9f5503ac921c
🧪 View experiment at: https://dagshub.com/rahulpatel16092005/youtube-comment-analysis.mlflow/#/experiments/1


[I 2026-04-13 16:41:58,718] Trial 0 finished with value: 0.6183008318559935 and parameters: {'vectorizer_type': 'word2vec', 'vector_size': 168}. Best is trial 0 with value: 0.6183008318559935.


🏃 View run (1, 2)_bow_6268 at: https://dagshub.com/rahulpatel16092005/youtube-comment-analysis.mlflow/#/experiments/1/runs/6161068e849347f483e3e69ec8a66da5
🧪 View experiment at: https://dagshub.com/rahulpatel16092005/youtube-comment-analysis.mlflow/#/experiments/1


[I 2026-04-13 16:46:37,761] Trial 1 finished with value: 0.6098459020864585 and parameters: {'vectorizer_type': 'bow', 'ngram_range': '(1, 2)', 'max_features': 6268}. Best is trial 0 with value: 0.6183008318559935.


...

[I 2026-04-13 18:53:25,803] Trial 33 finished with value: 0.6177553525160234 and parameters: {'vectorizer_type': 'tfidf', 'ngram_range': '(1, 2)', 'max_features': 1951}. Best is trial 12 with value: 0.6414837038047184.


🏃 View run (1, 2)_tfidf_1709 at: https://dagshub.com/rahulpatel16092005/youtube-comment-analysis.mlflow/#/experiments/1/runs/86ce9457cafa4a63ad5edf478516ed5c
🧪 View experiment at: https://dagshub.com/rahulpatel16092005/youtube-comment-analysis.mlflow/#/experiments/1


[I 2026-04-13 18:56:10,105] Trial 34 finished with value: 0.6228010364107459 and parameters: {'vectorizer_type': 'tfidf', 'ngram_range': '(1, 2)', 'max_features': 1709}. Best is trial 12 with value: 0.6414837038047184.


🏃 View run (1, 2)_tfidf_2604 at: https://dagshub.com/rahulpatel16092005/youtube-comment-analysis.mlflow/#/experiments/1/runs/c35449c5f05f40ce8b1f187f876412a3
🧪 View experiment at: https://dagshub.com/rahulpatel16092005/youtube-comment-analysis.mlflow/#/experiments/1


[I 2026-04-13 18:59:35,164] Trial 35 finished with value: 0.613800627301241 and parameters: {'vectorizer_type': 'tfidf', 'ngram_range': '(1, 2)', 'max_features': 2604}. Best is trial 12 with value: 0.6414837038047184.
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [ ]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')